# EOP Lab 2: Claim-contingent Disclosure

**Series**: Agentic AI Science Playbook -- EOP Domain
**Goal**: Given scientific claims of varying strength, determine the required disclosure scope.
**Prerequisites**: EOP Lab 1.
**Time**: ~45 min.

---

## Background: Claim Strength and Disclosure

Not all scientific claims require the same level of disclosure:

| Claim Type | Example | Required Disclosure |
|-----------|---------|---------------------|
| **Existential** | "We can produce X" | Basic code + inputs demonstrating X is possible |
| **Comparative** | "Our method outperforms baseline B" | Full pipeline + baseline reproduction |
| **Distributional** | "We reliably produce X (p<0.05 across N runs)" | Full pipeline + statistical scripts + raw data |
| **Novel method** | "We introduce algorithm A with properties P" | Algorithm implementation + formal verification |

The EOP agent assesses claim strength and recommends a corresponding disclosure level (1-4).

## Learning Objectives

By the end of this lab, you will be able to:
- Classify scientific claims by their strength (existential → novel method)
- Map claim types to appropriate disclosure levels (1-4)
- Build an agent that performs automated disclosure assessment
- Handle special cases like proprietary components

## Why This Matters

Not all scientific claims require the same evidence. Saying "we implemented X" requires less disclosure than "our method outperforms all baselines by 15% across 50 datasets."

**Claim-contingent disclosure** is the principle that **stronger claims require stronger evidence**. This lab builds an agent that:
- Reads a scientific claim
- Classifies its strength
- Determines exactly what computational artifacts must be disclosed

This is critical for:
- **Authors**: Know what you need to share before submission
- **Reviewers**: Know what to request based on claim type
- **Journals**: Establish tiered, fair disclosure policies

> **Key principle**: Disclosure requirements should be proportional to claim strength. This prevents both under-disclosure (weak evidence for strong claims) and over-burden (demanding full pipelines for simple existence claims).

### Install Dependencies
Install the required Python packages for this lab. We need `openai` for LLM access and `pydantic` for data validation of our tool schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Set up your OpenAI or NVIDIA NIM connection. This cell detects which API you have configured and creates the client. It also imports the core libraries we will use throughout the lab.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field
from typing import Literal

## Tool Schema: Claim Analysis

## Concept: The Disclosure Level Framework

The 4-level disclosure framework maps directly from claim strength to required artifacts:

```
Level 1 (Basic)         ← Existential claims ("X exists")
  └── Sample input + key code + README

Level 2 (Standard)      ← Comparative claims ("A > B")
  └── Full data + pipeline + outputs + figures + docs

Level 3 (Extended)      ← Distributional claims ("reliable across conditions")
  └── Level 2 + statistical scripts + raw data + configs

Level 4 (Comprehensive) ← Novel method claims ("we introduce algorithm A")
  └── Level 3 + unit tests + formal spec + all ECF types
```

This is designed to be **practical and incremental** — most papers only need Level 1 or 2. Level 4 is reserved for papers whose primary contribution IS the method.

### Define Claim Analysis Schemas
Two tools: `analyze_claim` classifies claim strength, `determine_disclosure` recommends the required disclosure level. Each tool has typed Pydantic arguments that the LLM will use to structure its function calls.

In [ ]:
ClaimType = Literal["existential", "comparative", "distributional", "novel_method", "descriptive"]

class AnalyzeClaimArgs(BaseModel):
    claim_text: str = Field(..., description="The scientific claim to analyze.")
    paper_context: str | None = Field(None, description="Brief description of the paper or study context.")

class DetermineDisclosureArgs(BaseModel):
    claim_text: str = Field(..., description="The scientific claim.")
    claim_type: ClaimType = Field(..., description="Type of claim.")
    has_proprietary_components: bool = Field(False, description="Whether the software has proprietary components.")

OPENAI_TOOLS = [
    {"type": "function", "function": {
        "name": "analyze_claim",
        "description": "Analyze a scientific claim and classify its type and strength.",
        "parameters": AnalyzeClaimArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "determine_disclosure",
        "description": "Determine the required disclosure scope for a given claim type.",
        "parameters": DetermineDisclosureArgs.model_json_schema()}},
]
SCHEMA_MAP = {"analyze_claim": AnalyzeClaimArgs, "determine_disclosure": DetermineDisclosureArgs}
print("Tools:", [t["function"]["name"] for t in OPENAI_TOOLS])

## Agent and Tool Implementations

### Build the Agent and Tool Implementations
The disclosure levels are mapped in a lookup table from claim type to required artifacts. The `analyze_claim` tool uses the LLM to classify claim type and strength. The `determine_disclosure` tool maps the classification to a specific disclosure level. The agent orchestrates both tools in sequence.

In [ ]:
DISCLOSURE_LEVELS = {
    "existential":   {"level": 1, "label": "Basic",        "requirements": ["input_data (sample)", "experimental_process (key steps)", "documentation"]},
    "descriptive":   {"level": 1, "label": "Basic",        "requirements": ["input_data (sample)", "documentation"]},
    "comparative":   {"level": 2, "label": "Standard",     "requirements": ["full input_data", "experimental_process", "output_data", "visual_claim", "documentation"]},
    "distributional":{"level": 3, "label": "Extended",     "requirements": ["full input_data", "experimental_process", "output_data", "visual_data", "plotting_process", "visual_claim", "statistical_scripts", "documentation"]},
    "novel_method":  {"level": 4, "label": "Comprehensive","requirements": ["full pipeline (all 7 ECF types)", "algorithm implementation", "unit tests", "formal specification", "documentation"]},
}

def execute_analyze_claim(args: AnalyzeClaimArgs) -> dict:
    prompt = (
        f"Classify this scientific claim into one of: existential, comparative, distributional, novel_method, descriptive.\n"
        f"Claim: {args.claim_text}\n"
        f"Context: {args.paper_context or 'not specified'}\n"
        f"Return JSON: {{\"claim_type\": ..., \"reasoning\": ..., \"strength_score\": 1-5}}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=200,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return json.loads(r.choices[0].message.content or "{}")

def execute_determine_disclosure(args: DetermineDisclosureArgs) -> str:
    info = DISCLOSURE_LEVELS.get(args.claim_type, DISCLOSURE_LEVELS["existential"])
    proprietary_note = (
        "\n  NOTE: Proprietary components present -- consider providing auditable intermediate data "
        "or a functional equivalent under restricted access."
    ) if args.has_proprietary_components else ""
    return (
        f"[determine_disclosure]\n"
        f"  Claim: {args.claim_text[:80]!r}\n"
        f"  Type: {args.claim_type} -> Disclosure Level {info['level']} ({info['label']})\n"
        f"  Required artifacts:\n" +
        "\n".join(f"    - {req}" for req in info["requirements"]) +
        proprietary_note
    )

def run_tool(name, args):
    if name == "analyze_claim": return execute_analyze_claim(args)
    if name == "determine_disclosure": return execute_determine_disclosure(args)
    return f"[error] Unknown: {name}"

EOP_SYSTEM = ("You are an EOP disclosure advisor. Analyze scientific claims and determine "
              "required disclosure scope using the provided tools.")

def agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": EOP_SYSTEM},
                   {"role": "user", "content": user_message}],
        tools=OPENAI_TOOLS, tool_choice="auto")
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "content": msg.content}
    call = msg.tool_calls[0]
    validated = SCHEMA_MAP[call.function.name](**json.loads(call.function.arguments))
    return {"tool": call.function.name, "args": validated}

## Experiments: Claims of Varying Strength

### Test with Claims of Increasing Strength
We test 4 claims ranging from existential ("we can produce X") to novel method ("we introduce algorithm A"). Watch how disclosure requirements escalate with claim strength. For each claim, the agent first analyzes the claim type, then determines the corresponding disclosure level.

In [ ]:
test_claims = [
    "Our implementation can generate valid protein structures.",
    "Our model achieves 92% accuracy, outperforming the SOTA baseline by 4%.",
    "Our algorithm reliably reduces energy consumption by 30% (p<0.01, N=50 runs across 3 datasets).",
    "We introduce a novel graph neural network architecture with provably lower sample complexity.",
]

for claim in test_claims:
    print(f"Claim: {claim[:80]}")

    # Step 1: Analyze claim type
    r1 = agent(f"Analyze this claim: {claim}")
    if r1["tool"]:
        analysis = run_tool(r1["tool"], r1["args"])
        claim_type = analysis.get("claim_type", "existential")
        print(f"  Analysis: {analysis}")

        # Step 2: Determine disclosure
        r2 = agent(
            f"Determine the required disclosure scope for this {claim_type} claim: {claim}")
        if r2["tool"]:
            print(run_tool(r2["tool"], r2["args"]))
    print()

<details>
<summary>Expected output (click to expand)</summary>

```
Claim: Our implementation can generate valid protein structures.
  Analysis: {'claim_type': 'existential', 'reasoning': '...', 'strength_score': 2}
[determine_disclosure]
  Claim: 'Our implementation can generate valid protein structures.'
  Type: existential -> Disclosure Level 1 (Basic)
  Required artifacts:
    - input_data (sample)
    - experimental_process (key steps)
    - documentation

Claim: Our model achieves 92% accuracy, outperforming the SOTA baseline by 4%.
  Analysis: {'claim_type': 'comparative', 'reasoning': '...', 'strength_score': 3}
[determine_disclosure]
  Claim: 'Our model achieves 92% accuracy, outperforming the SOTA baseline by 4%'
  Type: comparative -> Disclosure Level 2 (Standard)
  Required artifacts:
    - full input_data
    - experimental_process
    - output_data
    - visual_claim
    - documentation

Claim: Our algorithm reliably reduces energy consumption by 30% (p<0.01, N=50 ...
  Analysis: {'claim_type': 'distributional', 'reasoning': '...', 'strength_score': 4}
[determine_disclosure]
  Claim: 'Our algorithm reliably reduces energy consumption by 30% (p<0.01, N=5'
  Type: distributional -> Disclosure Level 3 (Extended)
  Required artifacts:
    - full input_data
    - experimental_process
    - output_data
    - visual_data
    - plotting_process
    - visual_claim
    - statistical_scripts
    - documentation

Claim: We introduce a novel graph neural network architecture with provably low...
  Analysis: {'claim_type': 'novel_method', 'reasoning': '...', 'strength_score': 5}
[determine_disclosure]
  Claim: 'We introduce a novel graph neural network architecture with provably l'
  Type: novel_method -> Disclosure Level 4 (Comprehensive)
  Required artifacts:
    - full pipeline (all 7 ECF types)
    - algorithm implementation
    - unit tests
    - formal specification
    - documentation
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Test: Proprietary Components
What happens when a claim involves proprietary software? The agent should recommend alternative disclosure strategies like auditable intermediate data. This tests the `has_proprietary_components` flag in the disclosure tool, which adds special handling for commercial or restricted-access software.

In [ ]:
# Claim with proprietary components
claim = "Our drug discovery pipeline identifies novel lead compounds with 40% higher hit rate."
r1 = agent(f"Analyze this claim: {claim}")
if r1["tool"]:
    analysis = run_tool(r1["tool"], r1["args"])
    claim_type = analysis.get("claim_type", "distributional")
r2 = agent(
    f"Determine disclosure for this {claim_type} claim with proprietary ML model: {claim}")
if r2["tool"]:
    print(run_tool(r2["tool"], r2["args"]))

<details>
<summary>Expected output (click to expand)</summary>

```
[determine_disclosure]
  Claim: 'Our drug discovery pipeline identifies novel lead compounds with 40% h'
  Type: distributional -> Disclosure Level 3 (Extended)
  Required artifacts:
    - full input_data
    - experimental_process
    - output_data
    - visual_data
    - plotting_process
    - visual_claim
    - statistical_scripts
    - documentation
  NOTE: Proprietary components present -- consider providing auditable intermediate data
  or a functional equivalent under restricted access.
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **Classify one of your own recent claims.** Is it existential, comparative, distributional, or novel method? What disclosure level does it require?
2. **The agent handled proprietary components** by suggesting "auditable intermediate data." What other strategies exist for disclosing proprietary research?
3. **How would claim-contingent disclosure change** the way you structure a paper's supplementary materials?

## Summary

| Claim Type | Level | Key Requirement |
|-----------|-------|----------------|
| Existential | 1 | Sample inputs + key code steps |
| Comparative | 2 | Full pipeline + baseline |
| Distributional | 3 | Full pipeline + statistics + raw data |
| Novel method | 4 | All ECF types + unit tests + spec |

**Next**: EOP Lab 3 -- advocacy across different audiences with LLM-as-judge evaluation.

---
*Agentic AI Science Playbook -- EOP Domain, Lab EOP2.*